# WNBA intro — sportsdataverse-py

A focused tour of the `sdv.wnba` submodule: teams, rosters, schedules, play-by-play, player and team season stats, standings, the draft, and the `load_wnba_*` parquet loaders. Most calls hit ESPN's public API and return tidy [polars](https://pola.rs) frames.

R companion: [wehoop](https://wehoop.sportsdataverse.org). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Teams

`espn_wnba_teams()` returns one row per franchise. The id, location, name, and abbreviation are the keys you'll reuse to fetch rosters, schedules, and stats.

In [ ]:
teams = sdv.wnba.espn_wnba_teams()
teams.shape

In [ ]:
teams.select([
    'team_id', 'team_location', 'team_name',
    'team_abbreviation', 'team_display_name'
]).head(10)

## Team roster — Las Vegas Aces

`espn_wnba_team_roster(team_id=..., season=...)` lists active players for one team. The Aces are `team_id=17`. Player columns are unprefixed (`athlete_id`, `full_name`, `jersey`, `position_abbreviation`).

In [ ]:
aces_roster = sdv.wnba.espn_wnba_team_roster(team_id=17, season=2024)
aces_roster.select([
    'athlete_id', 'full_name', 'jersey',
    'position_abbreviation', 'display_height', 'age'
]).head(12)

## Schedule

`espn_wnba_schedule(dates=YYYYMMDD)` pulls one day; pass a `'YYYYMMDD-YYYYMMDD'` string for a range. Team-name columns are `home_display_name` / `away_display_name`, and `home_score` / `away_score` come back as **strings** — cast before doing arithmetic.

The single date below (Oct 20, 2024) is Game 5 of the 2024 WNBA Finals.

In [ ]:
one_day = sdv.wnba.espn_wnba_schedule(dates=20241020)
one_day.select([
    'id', 'home_display_name', 'away_display_name',
    'home_score', 'away_score', 'status_type_description'
])

### Schedule over a date range

Cast the score strings to integers and derive a winning-margin column to show a small polars transform.

In [ ]:
finals = sdv.wnba.espn_wnba_schedule(dates='20241016-20241020')
finals.select([
    'id', 'home_display_name', 'away_display_name', 'home_score', 'away_score'
]).with_columns([
    pl.col('home_score').cast(pl.Int64, strict=False).alias('home_pts'),
    pl.col('away_score').cast(pl.Int64, strict=False).alias('away_pts'),
]).with_columns(
    (pl.col('home_pts') - pl.col('away_pts')).abs().alias('margin')
)

## Play-by-play — 2024 WNBA Finals Game 5

`espn_wnba_pbp(game_id=...)` returns a **dict** of component pieces (`plays`, `boxscore`, `header`, `winprobability`, ...). The `plays` entry is a list of raw ESPN dicts; build a frame with `pl.DataFrame(..., infer_schema_length=None)`. Its columns use raw **dot-notation** (`period.number`, `clock.displayValue`, `scoringPlay`, `type.text`).

In [ ]:
pbp = sdv.wnba.espn_wnba_pbp(game_id=401726992)
list(pbp.keys())[:10]

In [ ]:
plays = pl.DataFrame(pbp['plays'], infer_schema_length=None)
plays.select([
    'period.number', 'clock.displayValue', 'type.text', 'text', 'scoringPlay'
]).head(10)

Filter to scoring plays only to see how the lead changed.

In [ ]:
(plays
    .filter(pl.col('scoringPlay'))
    .select(['period.number', 'clock.displayValue', 'homeScore', 'awayScore', 'text'])
    .tail(8))

## Player season stats — Caitlin Clark

`espn_wnba_player_stats(athlete_id=..., season=...)` works for the WNBA and returns a single wide row covering ESPN's `general` / `offensive` / `defensive` stat groups (averages and totals). Caitlin Clark is `athlete_id=4433403`.

In [ ]:
cc = sdv.wnba.espn_wnba_player_stats(athlete_id=4433403, season=2024)
cc.shape

In [ ]:
cc.select([
    'full_name', 'team_abbreviation', 'general_games_played',
    'offensive_avg_points', 'offensive_avg_assists',
    'general_avg_rebounds', 'offensive_three_point_field_goal_pct'
])

## Team season stats

`espn_wnba_team_stats(team_id=..., season=...)` returns a **dict** keyed by category: `{'Averages', 'Totals', 'Misc'}`. Each value is a long frame of `stat_name` / `display_value` rows — index into the dict, don't call `.head()` on it directly.

In [ ]:
aces_stats = sdv.wnba.espn_wnba_team_stats(team_id=17, season=2024)
list(aces_stats.keys())

In [ ]:
aces_stats['Averages'].select(['stat_name', 'abbreviation', 'display_value']).head(10)

## Standings

`espn_wnba_standings(season=...)` returns one row per team with wins, losses, win percentage, and point differential.

In [ ]:
standings = sdv.wnba.espn_wnba_standings(season=2024)
(standings
    .select(['team_display_name', 'wins', 'losses', 'win_percent', 'point_differential'])
    .sort('win_percent', descending=True)
    .head(8))

## Draft

`espn_wnba_draft(season=...)` lists every pick. The 2024 draft headlined with Caitlin Clark going first overall to the Indiana Fever.

In [ ]:
draft = sdv.wnba.espn_wnba_draft(season=2024)
draft.select([
    'overall_pick', 'team_display_name',
    'athlete_display_name', 'athlete_position_abbreviation', 'school_name'
]).head(10)

## Bulk loaders (`load_wnba_*`)

The `load_wnba_*` functions read pre-built parquet releases (whole seasons at once) instead of calling the live API per game. They return polars frames; pass `return_as_pandas=True` for pandas. Available loaders include `load_wnba_schedule`, `load_wnba_pbp`, `load_wnba_player_boxscore`, `load_wnba_team_boxscore`, `load_wnba_player_season_stats`, `load_wnba_rosters`, `load_wnba_standings`, and `load_wnba_shots`.

In [ ]:
sched_2024 = sdv.wnba.load_wnba_schedule(seasons=[2024])
sched_2024.shape

In [ ]:
box_2024 = sdv.wnba.load_wnba_player_boxscore(seasons=[2024])
box_2024.select([
    'game_id', 'game_date', 'athlete_display_name',
    'team_abbreviation', 'minutes', 'points', 'rebounds', 'assists'
]).head()

## Pipeline example: top 10 WNBA scorers of 2024

Load the full-season player boxscore parquet, drop did-not-play rows, then aggregate points and assists per player with polars.

In [ ]:
top_scorers = (
    box_2024
    .filter(~pl.col('did_not_play'))
    .group_by(['athlete_display_name', 'team_abbreviation'])
    .agg([
        pl.len().alias('games'),
        pl.col('points').sum().alias('total_points'),
        pl.col('points').mean().round(1).alias('ppg'),
        pl.col('assists').mean().round(1).alias('apg'),
    ])
    .filter(pl.col('games') >= 20)
    .sort('ppg', descending=True)
    .head(10)
)
top_scorers

## Cross-references

- R companion: [wehoop](https://wehoop.sportsdataverse.org)
- Data source: ESPN, WNBA Stats API
- Stats-API alternative (Python): [nba_api](https://github.com/swar/nba_api) (also covers the WNBA)
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/wnba/index.md`
- Next notebook: `09_mlb_intro.ipynb`